In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [ ]:
idhp_path = Path("/Users/vzuev/Documents/git/git_other/CEVAE/datasets/IHDP")
ihdp_cols = [s[:-1] for s in np.loadtxt(idhp_path / "columns.txt", dtype=str)][:-2]
ihdp_cols.extend([f"x{i}" for i in range(2, 26)])
ihdp_cols[:7]


In [ ]:
csvs = []
for csv_path in (idhp_path / "csv").glob("*.csv"):
    csvs.append(pd.read_csv(csv_path, header=None))
data = pd.concat(csvs)
data.columns = ihdp_cols

y_col_name = "delta_y"
data[y_col_name] = (data["y_cfactual"] - data["y_factual"]) * (-1) ** data["treatment"]
data

In [ ]:
exclude_cols = ["treatment", "y_cfactual", "y_factual"]
data_x, data_y = data.drop(columns=[*exclude_cols, y_col_name]), data[y_col_name]
data_x  # categorical features are already encoded as ordinals

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

train_x, test_x, train_y, test_y = train_test_split(data_x, data_y, random_state=0)
train_y = pd.DataFrame(train_y)

In [10]:
from mothernet.prediction.mothernet_additive import MotherNetAdditiveRegressor
from mothernet.utils import get_mn_model

model_path = get_mn_model("baam_Daverage_l1e-05_maxnumclasses0_nsamples500_numfeatures10_yencoderlinear_05_08_2024_03_04_01_epoch_40.cpkt")
reg = MotherNetAdditiveRegressor(device="cpu", path=model_path)

In [20]:
import numpy as np

ss = StandardScaler().fit(train_y)
reg.fit(np.array(train_x[:300]), ss.transform(train_y[:300]))  # more than 3m on CPU for ~5600 train samples - aborting; TODO try GPU

In [ ]:
import matplotlib.pyplot as plt

y_pred = reg.predict(test_x)
plt.plot(test_y, ss.inverse_transform(y_pred.reshape(-1, 1)), 'o')
plt.xlabel("test_y")
plt.ylabel("y_pred")